In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/netflix_titles.csv")
print(df.shape)
df.head()

(8807, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB


In [3]:
print("Fully duplicated rows:", df.duplicated().sum())
print("Duplicated show_id:", df["show_id"].duplicated().sum())
print("Duplicated title+type+release_year:", df.duplicated(subset=["title", "type", "release_year"]).sum())
print()
print(df["rating"].value_counts(dropna=False))

Fully duplicated rows: 0
Duplicated show_id: 0
Duplicated title+type+release_year: 0

rating
TV-MA       3207
TV-14       2160
TV-PG        863
R            799
PG-13        490
TV-Y7        334
TV-Y         307
PG           287
TV-G         220
NR            80
G             41
TV-Y7-FV       6
NaN            4
NC-17          3
UR             3
74 min         1
84 min         1
66 min         1
Name: count, dtype: int64


In [4]:
bad = df[df["rating"].str.contains("min", na=False)]
bad[["show_id", "title", "type", "rating", "duration"]]

,show_id,title,type,rating,duration
5541,s5542,Louis C.K. 2017,Movie,74 min,NaN
5794,s5795,Louis C.K.: Hilarious,Movie,84 min,NaN
5813,s5814,Louis C.K.: Live at the Comedy Store,Movie,66 min,NaN


In [5]:
print(df["type"].value_counts(dropna=False))
print()
print(df["release_year"].describe())
print()
print(df["date_added"].head(10).tolist())
print()
print(df["country"].value_counts().head(10))

type
Movie      6131
TV Show    2676
Name: count, dtype: int64

count    8807.000000
mean     2014.180198
std         8.819312
min      1925.000000
25%      2013.000000
50%      2017.000000
75%      2019.000000
max      2021.000000
Name: release_year, dtype: float64

['September 25, 2021', 'September 24, 2021', 'September 24, 2021', 'September 24, 2021', 'September 24, 2021', 'September 24, 2021', 'September 24, 2021', 'September 24, 2021', 'September 24, 2021', 'September 24, 2021']

country
United States     2818
India              972
United Kingdom     419
Japan              245
South Korea        199
Canada             181
Spain              145
France             124
Mexico             110
Egypt              106
Name: count, dtype: int64


In [6]:
text_cols = [c for c in df.columns if c != "release_year"]
for col in text_cols:
    s = df[col].dropna()
    print(col, "-> values with extra spaces:", (s != s.str.strip()).sum())

print()
print("Country cells with a comma:", df["country"].str.contains(",", na=False).sum())
print(df.loc[df["country"].str.contains(",", na=False), "country"].head(5).tolist())

print()
parsed = pd.to_datetime(df["date_added"].str.strip(), format="%B %d, %Y", errors="coerce")
print("date_added that fail to parse:", parsed.isna().sum() - df["date_added"].isna().sum())

show_id -> values with extra spaces: 0
type -> values with extra spaces: 0
title -> values with extra spaces: 1
director -> values with extra spaces: 0
cast -> values with extra spaces: 0
country -> values with extra spaces: 0
date_added -> values with extra spaces: 88
rating -> values with extra spaces: 0
duration -> values with extra spaces: 0
listed_in -> values with extra spaces: 0
description -> values with extra spaces: 0

Country cells with a comma: 1320
['United States, Ghana, Burkina Faso, United Kingdom, Germany, Ethiopia', 'Germany, Czech Republic', 'United States, India, France', 'China, Canada, United States', 'South Africa, United States, Japan']

date_added that fail to parse: 0


In [7]:
print(df.groupby("type")["duration"].apply(lambda s: s.str.extract(r"(\D+)$")[0].str.strip().value_counts()))

type     0      
Movie    min        6128
TV Show  Season     1793
         Seasons     883
Name: duration, dtype: int64


In [8]:
raw = pd.read_csv("../data/raw/netflix_titles.csv")
pattern = r"^\s*,|,\s*$|,\s*,"
raw.loc[raw["country"].str.contains(pattern, na=False), ["show_id", "title", "country"]]

,show_id,title,country
193,s194,D.P.,", South Korea"
365,s366,Eyes of a Thief,", France, Algeria"
1192,s1193,The Present,"United Kingdom,"
2224,s2225,Funan,"France, Belgium, Luxembourg, Cambodia,"
4653,s4654,City of Joy,"United States,"
5925,s5926,Virunga,"United Kingdom,"
7007,s7008,Hotel Pacific,"Poland,"
